# CRISP-DM Phase 4: Modelling — Model 1, Track A
## Random Forest Baseline — Selected Features (k=7)
Loads the pre-selected feature sets produced by feature-selection.ipynb.
Pipeline: SelectKBest output → StandardScaler → RandomForestClassifier (default params)
Baseline only — default parameters, no tuning. Establishes the RF performance
floor on the k=7 feature subset before hyperparameter optimisation.

In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report, f1_score

# Load pre-selected feature sets from feature-selection.ipynb
X_train = joblib.load('../data/X_train_sel.pkl')
X_test  = joblib.load('../data/X_test_sel.pkl')
y_train = joblib.load('../data/y_train.pkl')
y_test  = joblib.load('../data/y_test.pkl')

X_train = X_train.copy()
X_test  = X_test.copy()

print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Features: {list(X_train.columns)}")

Train: (4800, 7) | Test: (1200, 7)
Features: ['yrs_experience', 'income', 'mortgage_amt', 'fixed_deposit_acct', 'education_level_Advanced or Professional', 'education_level_Graduate', 'education_level_Undergraduate']


## Standardisation — Per Column, Fit on Training Data Only
StandardScaler is fitted exclusively on training data.
Applying to the test set uses transform() only — never fit_transform() —
to prevent test-set statistics from contaminating the model (data leakage).
The .reshape(-1, 1) is required because StandardScaler expects a 2D array;
a single Pandas column is 1D and must be reshaped before transformation.

In [2]:
continuous_cols = [col for col in X_train.columns
                   if col in ['age', 'yrs_experience', 'family_size',
                               'income', 'mortgage_amt', 'credit_card_spend']]

scaler = StandardScaler()

for col in continuous_cols:
    scaler.fit(X_train[col].values.reshape(-1, 1))
    X_train[col] = scaler.transform(X_train[col].values.reshape(-1, 1))
    X_test[col]  = scaler.transform(X_test[col].values.reshape(-1, 1))

print(f"Scaled columns: {continuous_cols}")

Scaled columns: ['yrs_experience', 'income', 'mortgage_amt']


## Random Forest Baseline — Default Parameters
All parameters left at sklearn defaults to establish a reproducible baseline.
random_state=42 ensures identical results on each run.
F1-Score is used as the primary metric due to class imbalance (~85/15 split).
Accuracy is discarded as a metric — a model predicting 'no' for every customer
would achieve 85% accuracy without any predictive value.

In [3]:
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
precision   = tp / (tp + fp)
recall      = tp / (tp + fn)
f1          = 2 * (precision * recall) / (precision + recall)
specificity = tn / (tn + fp)

print("=== RF Baseline — Selected Features (k=7) ===")
print(f"\nConfusion Matrix:")
print(f"  TN: {tn}  FP: {fp}")
print(f"  FN: {fn}  TP: {tp}")
print(f"\nPrecision:   {precision:.4f}")
print(f"Recall:      {recall:.4f}")
print(f"F1-Score:    {f1:.4f}")
print(f"Specificity: {specificity:.4f}")
print(f"\nFull classification report:")
print(classification_report(y_test, y_pred))

=== RF Baseline — Selected Features (k=7) ===

Confusion Matrix:
  TN: 967  FP: 53
  FN: 58  TP: 122

Precision:   0.6971
Recall:      0.6778
F1-Score:    0.6873
Specificity: 0.9480

Full classification report:
              precision    recall  f1-score   support

           0       0.94      0.95      0.95      1020
           1       0.70      0.68      0.69       180

    accuracy                           0.91      1200
   macro avg       0.82      0.81      0.82      1200
weighted avg       0.91      0.91      0.91      1200

